In [3]:
import pandas as pd
import numpy as np
from itertools import combinations
from collections import Counter

In [2]:
data=pd.read_csv(r"C:\Users\Pravalika.b\Downloads\Online Retail.csv")
data.head

<bound method NDFrame.head of        InvoiceNo StockCode                          Description  Quantity  \
0         536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1         536365     71053                  WHITE METAL LANTERN         6   
2         536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3         536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4         536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   
...          ...       ...                                  ...       ...   
541904    581587     22613          PACK OF 20 SPACEBOY NAPKINS        12   
541905    581587     22899         CHILDREN'S APRON DOLLY GIRL          6   
541906    581587     23254        CHILDRENS CUTLERY DOLLY GIRL          4   
541907    581587     23255      CHILDRENS CUTLERY CIRCUS PARADE         4   
541908    581587     22138        BAKING SET 9 PIECE RETROSPOT          3   

          InvoiceDate  UnitPrice  CustomerID 

In [4]:
data.columns

Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country'],
      dtype='object')

In [5]:
## Remove cancelled orders
data=data[data["InvoiceNo"].astype(str).str[0]!="C"]

In [7]:
## Remove missing customer ID and description
data=data.dropna(subset=["CustomerID","Description"])

In [11]:
## Remove duplicate products within the same order
order_products=(
    data.groupby("InvoiceNo")["Description"]
    .apply(lambda x: sorted(set(x)))
)
print("Number of Orders:", len(order_products))

Number of Orders: 18536


In [13]:
## cerate product pairs
pair_counter=Counter()
for products in order_products:
    if len(products)>=2:
        pairs=combinations(products,2)
        pair_counter.update(pairs)

In [14]:
## convert pairs into dataframe
pairs_df=pd.DataFrame(
    pair_counter.items(),
    columns=["Product Pair","Order Count"]
)

In [17]:
## Split product pair into two columns
pairs_df[["Product 1","Product 2"]]=pd.DataFrame(
    pairs_df["Product Pair"].tolist(),
    index=pairs_df.index
)
pairs_df=pairs_df.drop(columns=["Product Pair"])

In [19]:
## calxulate support %
total_orders=len(order_products)
pairs_df["Supprt %"]=(
    pairs_df["Order Count"] / total_orders*100).round(2)

In [21]:
## sort by most frequently purchased together
pairs_df=pairs_df.sort_values(
    "Order Count",
    ascending=False
).reset_index(drop=True)

In [22]:
## display top 20 pairs
print("\n Top 20 frequently purchased together:")
print(pairs_df.head(20))


 Top 20 frequently purchased together:
    Order Count                           Product 1  \
0           546             JUMBO BAG PINK POLKADOT   
1           541     GREEN REGENCY TEACUP AND SAUCER   
2           530          ALARM CLOCK BAKELIKE GREEN   
3           523             LUNCH BAG PINK POLKADOT   
4           517             LUNCH BAG  BLACK SKULL.   
5           468         WOODEN FRAME ANTIQUE WHITE    
6           467             LUNCH BAG RED RETROSPOT   
7           464             LUNCH BAG  BLACK SKULL.   
8           463  GARDENERS KNEELING PAD CUP OF TEA    
9           460     GREEN REGENCY TEACUP AND SAUCER   
10          458                 LUNCH BAG CARS BLUE   
11          455    RED HANGING HEART T-LIGHT HOLDER   
12          451     PAPER CHAIN KIT 50'S CHRISTMAS    
13          450             LUNCH BAG RED RETROSPOT   
14          436      PINK REGENCY TEACUP AND SAUCER   
15          434             JUMBO BAG RED RETROSPOT   
16          433          

In [23]:
## Save result
pairs_df.to_csv(
    "frequently_purchased_together.csv",
    index=False
)
print("\n Analysis completed sucessfully!")


 Analysis completed sucessfully!
